# 서울시 전기차 충전기 가용률 리포트 뷰어

`0713/<구>/<시>.json` 파일을 읽어서 원래 콘솔 리포트와 같은 형태로 보여줍니다.

In [1]:
import json
import os
import glob

ROOT = "0713"

def load_report(gu_name, hour_label):
    """hour_label 예: '17시'"""
    path = os.path.join(ROOT, gu_name, f"{hour_label}.json")
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def print_report(data):
    s = data["summary"]
    gu = data["district"]
    print("=" * 50)
    print(f" \U0001f680 [\uc885\ud569 \ub9ac\ud3ec\ud2b8] \uc11c\uc6b8\uc2dc {gu} \uc804\uae30\ucc28 \uc778\ud504\ub77c \ud604\ud669 ({data.get('collected_at_kst','')[:16]})")
    print("=" * 50)
    print(f" \U0001f4ca {gu} \uc804\uccb4 \ucda9\uc804\uae30 \uac00\uc6a9\ub960 : {s['total_percent']}% ({s['available_count']}\ub300 \uc0ac\uc6a9 \uac00\ub2a5 / \ucd1d {s['total_count']}\ub300)")
    print(f" \U0001f539 50kW \ubbf8\ub9cc \uc644\uc18d \ucda9\uc804\uae30 \uac00\uc6a9\ub960: {s['slow_percent']}% ({s['slow_avail']}\ub300 \uc0ac\uc6a9 \uac00\ub2a5 / \ucd1d {s['slow_total']}\ub300)")
    print(f" \U0001f538 50kW \uc774\uc0c1 \uace0\uc18d \ucda9\uc804\uae30 \uac00\uc6a9\ub960: {s['fast_percent']}% ({s['fast_avail']}\ub300 \uc0ac\uc6a9 \uac00\ub2a5 / \ucd1d {s['fast_total']}\ub300)")
    print("=" * 50)
    print()
    print("=" * 50)
    print(f"\U0001f3e2 \uad00\ub0b4 \uac1c\ubcc4 \ucda9\uc804\uc18c\ubcc4 \uc0c1\uc138 \uac00\uc6a9\uc131 \ub9ac\uc2a4\ud2b8 (\uc124\uce58 \ub300\uc218\uc21c \uc815\ub82c)")
    print("=" * 50)
    for idx, st in enumerate(data["stations"], start=1):
        print(f"[{idx}] \U0001f3e2 \ucda9\uc804\uc18c\uba85: {st['name']}")
        print(f" \U0001f7e2 \uc885\ud569 \uac00\uc6a9\ub960: {st['total_percent']}% ({st['available_count']}\ub300 \uac00\ub2a5 / \ucd1d {st['total_count']}\ub300)")
        print(f" \U0001f310 \uc8fc\uc18c: {st['address']}")
        print("-" * 50)

In [2]:
# 예시: 동대문구 17시 리포트 보기
data = load_report("동대문구", "17시")
print_report(data)

 🚀 [종합 리포트] 서울시 동대문구 전기차 인프라 현황 (2026-07-13T17:50)
 📊 동대문구 전체 충전기 가용률 : 80.2% (1790대 사용 가능 / 총 2231대)
 🔹 50kW 미만 완속 충전기 가용률: 80.6% (1715대 사용 가능 / 총 2127대)
 🔸 50kW 이상 고속 충전기 가용률: 72.1% (75대 사용 가능 / 총 104대)

🏢 관내 개별 충전소별 상세 가용성 리스트 (설치 대수순 정렬)
[1] 🏢 충전소명: 래미안엘리니티
 🟢 종합 가용률: 95.1% (58대 가능 / 총 61대)
 🌐 주소: 서울특별시 동대문구 한빛로 49
--------------------------------------------------
[2] 🏢 충전소명: 휘경주공1단지
 🟢 종합 가용률: 96.0% (48대 가능 / 총 50대)
 🌐 주소: 서울특별시 동대문구 한천로 248
--------------------------------------------------
[3] 🏢 충전소명: 이문아이파크자이 HDC 비거주 지하3층
 🟢 종합 가용률: 0.0% (0대 가능 / 총 44대)
 🌐 주소: 서울특별시 동대문구 이문동 149-7
--------------------------------------------------
[4] 🏢 충전소명: 래미안장안2차아파트
 🟢 종합 가용률: 88.1% (37대 가능 / 총 42대)
 🌐 주소: 서울특별시 동대문구 장안벚꽃로 167
--------------------------------------------------
[5] 🏢 충전소명: 청솔우성아파트
 🟢 종합 가용률: 100.0% (40대 가능 / 총 40대)
 🌐 주소: 서울특별시 동대문구 전농로10길 20
--------------------------------------------------
[6] 🏢 충전소명: 서울 동대문구 청량리역롯데캐슬SKY-L65
 🟢 종합 가용률: 94.9% (37대 가능 / 총 39대)
 🌐 주소: 서울특별시

In [3]:
# 특정 시간의 25개구 전체 요약을 한눈에 비교
import pandas as pd

def summary_table(hour_label):
    rows = []
    for gu_dir in sorted(glob.glob(os.path.join(ROOT, "*"))):
        gu_name = os.path.basename(gu_dir)
        path = os.path.join(gu_dir, f"{hour_label}.json")
        if not os.path.exists(path):
            continue
        with open(path, encoding="utf-8") as f:
            d = json.load(f)
        s = d["summary"]
        rows.append({
            "구": gu_name,
            "전체가용률(%)": s["total_percent"],
            "완속가용률(%)": s["slow_percent"],
            "고속가용률(%)": s["fast_percent"],
            "총충전기수": s["total_count"],
        })
    return pd.DataFrame(rows).sort_values("구").reset_index(drop=True)

summary_table("17시")

,구,전체가용률(%),완속가용률(%),고속가용률(%),총충전기수
0,강남구,78.7,78.5,82.0,5920
1,강동구,82.9,83.0,80.9,2966
2,강북구,87.2,88.9,75.9,1267
3,강서구,81.1,80.9,84.0,4820
4,관악구,85.6,85.5,85.7,1690
5,광진구,79.8,79.5,83.2,1493
6,구로구,80.5,80.7,78.0,3618
7,금천구,74.9,74.9,75.5,2258
8,노원구,82.9,83.2,78.3,3887
9,도봉구,79.5,81.1,68.3,1836


In [4]:
# 특정 구의 24시간 추이 보기 (데이터가 쌓이면 사용)
def district_timeseries(gu_name):
    rows = []
    for path in sorted(glob.glob(os.path.join(ROOT, gu_name, "*.json"))):
        hour_label = os.path.splitext(os.path.basename(path))[0]
        with open(path, encoding="utf-8") as f:
            d = json.load(f)
        s = d["summary"]
        rows.append({"시간": hour_label, "전체가용률(%)": s["total_percent"]})
    return pd.DataFrame(rows)

district_timeseries("동대문구")

,시간,전체가용률(%)
0,17시,80.2
